In [ ]:
# ============================================================================
# STATISTICAL VALIDATION - 30 COLD START RUNS
# ============================================================================

import cupy as cp
import numpy as np
import pickle
import time
import os
import json
import glob
from datetime import datetime
from google.colab import drive
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats

# Mount drive
drive.mount('/content/drive')

print("="*70)
print("📊 STATISTICAL VALIDATION - 30 COLD START RUNS")
print("="*70)

# ============================================================================
# CONFIGURATION
# ============================================================================
N_VERTICES = 4096
CLIQUE_SIZE = 60
POP_SIZE = 4096
N_RUNS = 30
MAX_ITERATIONS = 200000  # Reduced from 500k - enough to find optimum

# Algorithm parameters (proven)
MIGRATION_THRESHOLD = 100
DEEP_MUTATION_THRESHOLD = 250
EXPLORATION_PROB = 0.10
CANDIDATE_SAMPLES = 128
INITIAL_TEMPERATURE = 0.5
COOLING_RATE = 0.95
MIN_TEMPERATURE = 0.05
REHEAT_TEMPERATURE = 1.8
REHEAT_THRESHOLD = 4000

# Load Keller-6 data once
print(f"\n📊 Loading Keller-6 graph data...")
DATA_FILE = "/content/sample_data/keller_codes_d6.data"
with open(DATA_FILE, 'rb') as f:
    codes = pickle.load(f)

# Build adjacency matrix (shared across all runs)
n_words = (N_VERTICES + 63) // 64
adj_np = np.zeros((N_VERTICES, n_words), dtype=np.uint64)
for u, mask in enumerate(codes):
    for v in range(N_VERTICES):
        if (mask >> v) & 1:
            adj_np[u, v // 64] |= (np.uint64(1) << (v % 64))
adj_gpu = cp.array(adj_np)
print(f"✅ NAM loaded: {adj_np.nbytes/1e6:.2f} MB")

# ============================================================================
# CUDA KERNEL (same as before)
# ============================================================================
cuda_kernel = r'''
extern "C" __device__
unsigned int simple_rand(unsigned int *state) {
    *state ^= *state << 13;
    *state ^= *state >> 17;
    *state ^= *state << 5;
    return *state;
}

extern "C" __global__
void global_kernel(
    const unsigned long long* __restrict__ adj,
    int* __restrict__ pop,
    int* __restrict__ energies,
    int* __restrict__ stagnation,
    const float* __restrict__ rand_vals,
    float temperature,
    int best_idx,
    int clique_size,
    int n_words)
{
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid >= 4096) return;

    int base = tid * clique_size;
    unsigned int state = (unsigned int)(rand_vals[tid * 3] * 1234567.0f);

    if (stagnation[tid] > 100) {
        int best_base = best_idx * clique_size;
        for (int i = 0; i < clique_size; i++) {
            pop[base + i] = pop[best_base + i];
        }

        int num_mut = (stagnation[tid] > 250) ? 5 : 1;
        for (int m = 0; m < num_mut; m++) {
            int mut_pos = simple_rand(&state) % clique_size;
            int new_v;
            bool duplicate;
            do {
                new_v = simple_rand(&state) % 4096;
                duplicate = false;
                for (int i = 0; i < clique_size; i++) {
                    if (pop[base + i] == new_v) {
                        duplicate = true;
                        break;
                    }
                }
            } while (duplicate);
            pop[base + mut_pos] = new_v;
        }

        int new_energy = 0;
        for (int i = 0; i < clique_size; i++) {
            for (int j = i+1; j < clique_size; j++) {
                int u = pop[base + i];
                int v = pop[base + j];
                int word_idx = v >> 6;
                int bit_idx = v & 63;
                if ((adj[u * n_words + word_idx] >> bit_idx) & 1) {
                    new_energy++;
                }
            }
        }
        energies[tid] = new_energy;
        stagnation[tid] = 0;
        return;
    }

    int worst_pos = 0;
    int max_conflicts = -1;

    float r = rand_vals[tid * 3 + 2];
    if (r < 0.10f) {
        worst_pos = simple_rand(&state) % clique_size;
    } else {
        for (int i = 0; i < clique_size; i++) {
            int u = pop[base + i];
            int conflicts = 0;
            for (int j = 0; j < clique_size; j++) {
                if (i == j) continue;
                int v = pop[base + j];
                int word_idx = v >> 6;
                int bit_idx = v & 63;
                if ((adj[u * n_words + word_idx] >> bit_idx) & 1) {
                    conflicts++;
                }
            }
            if (conflicts > max_conflicts) {
                max_conflicts = conflicts;
                worst_pos = i;
            }
        }
    }

    int old_v = pop[base + worst_pos];

    int best_candidate = old_v;
    int best_candidate_conflicts = 999;

    for (int k = 0; k < 128; k++) {
        int cand = simple_rand(&state) % 4096;

        bool duplicate = false;
        for (int i = 0; i < clique_size; i++) {
            if (pop[base + i] == cand) {
                duplicate = true;
                break;
            }
        }
        if (duplicate) continue;

        int conf = 0;
        for (int j = 0; j < clique_size; j++) {
            if (j == worst_pos) continue;
            int v = pop[base + j];
            int word_idx = v >> 6;
            int bit_idx = v & 63;
            if ((adj[cand * n_words + word_idx] >> bit_idx) & 1) {
                conf++;
            }
        }

        if (conf < best_candidate_conflicts) {
            best_candidate_conflicts = conf;
            best_candidate = cand;
        }
        if (best_candidate_conflicts == 0) break;
    }

    int old_conflicts = 0;
    for (int j = 0; j < clique_size; j++) {
        if (j == worst_pos) continue;
        int v = pop[base + j];
        int word_idx = v >> 6;
        int bit_idx = v & 63;
        if ((adj[old_v * n_words + word_idx] >> bit_idx) & 1) {
            old_conflicts++;
        }
    }

    int delta = best_candidate_conflicts - old_conflicts;
    if (delta <= 0 || rand_vals[tid * 3 + 1] < expf(-(float)delta / temperature)) {
        pop[base + worst_pos] = best_candidate;
        energies[tid] += delta;
        stagnation[tid] = 0;
    } else {
        stagnation[tid]++;
    }
}
'''

# Compile kernel once
print("\n⚡ Compiling global kernel...")
module = cp.RawModule(code=cuda_kernel)
kernel = module.get_function("global_kernel")
print("✅ Kernel compiled")

# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================
def compute_energies_cpu(pop_cpu):
    energies = []
    for row in pop_cpu:
        e = 0
        for i in range(CLIQUE_SIZE):
            for j in range(i+1, CLIQUE_SIZE):
                u, v = row[i], row[j]
                word_idx = v >> 6
                bit_idx = v & 63
                if (adj_np[u, word_idx] >> np.uint64(bit_idx)) & np.uint64(1):
                    e += 1
        energies.append(e)
    return np.array(energies)

def compute_diversity(pop_cpu):
    """Compute population diversity metrics"""
    sample_size = min(100, len(pop_cpu))
    indices = np.random.choice(len(pop_cpu), sample_size, replace=False)
    sample = [pop_cpu[i] for i in indices]

    distances = []
    for i in range(len(sample)):
        set_i = set(sample[i])
        for j in range(i+1, len(sample)):
            set_j = set(sample[j])
            intersection = len(set_i & set_j)
            union = len(set_i | set_j)
            distances.append(1 - intersection/union if union > 0 else 0)

    return {
        'mean_pairwise_distance': float(np.mean(distances)) if distances else 0,
        'std_pairwise_distance': float(np.std(distances)) if distances else 0
    }

def clique_to_hash(vertices):
    """Create unique hash for a clique"""
    sorted_verts = tuple(sorted(vertices))
    import hashlib
    return hashlib.md5(str(sorted_verts).encode()).hexdigest()

# ============================================================================
# RUN SINGLE EXPERIMENT
# ============================================================================
def run_single_experiment(run_id, results_dir):
    """Execute one cold start run"""

    # Create run directory
    run_dir = f"{results_dir}/run_{run_id:03d}"
    os.makedirs(run_dir, exist_ok=True)

    # Initialize population (cold start - random)
    pop_cpu = np.zeros((POP_SIZE, CLIQUE_SIZE), dtype=np.int32)
    for i in range(POP_SIZE):
        pop_cpu[i] = np.random.choice(N_VERTICES, CLIQUE_SIZE, replace=False)

    energies_cpu = compute_energies_cpu(pop_cpu)
    pop_gpu = cp.array(pop_cpu, dtype=cp.int32)
    energies_gpu = cp.array(energies_cpu, dtype=cp.int32)
    stagnation_gpu = cp.zeros(POP_SIZE, dtype=cp.int32)

    # Tracking
    initial_best = int(np.min(energies_cpu))
    best_energy = initial_best
    best_clique = pop_cpu[np.argmin(energies_cpu)].tolist()
    convergence_iter = None
    found_cliques = set()

    # Run parameters
    temperature = INITIAL_TEMPERATURE
    stag_global = 0
    start_time = time.time()

    # Metrics
    history = []
    diversity_samples = []

    for iteration in range(MAX_ITERATIONS):
        # Generate random numbers
        rands = cp.random.random(POP_SIZE * 4, dtype=cp.float32)

        # Find global best
        e_np = energies_gpu.get()
        best_idx = int(np.argmin(e_np))

        # Launch kernel
        kernel(
            (32,), (128,),
            (adj_gpu, pop_gpu, energies_gpu, stagnation_gpu, rands,
             np.float32(temperature), best_idx, CLIQUE_SIZE, n_words)
        )

        # Periodic reporting
        if iteration % 500 == 0:
            e_np = energies_gpu.get()
            current_best = int(np.min(e_np))
            current_mean = float(np.mean(e_np))

            history.append({
                'iteration': iteration,
                'best': current_best,
                'mean': current_mean,
                'temperature': temperature
            })

            # Sample diversity every 5000 iterations
            if iteration % 5000 == 0:
                pop_sample = pop_gpu.get()[:100]
                diversity = compute_diversity(pop_sample)
                diversity['iteration'] = iteration
                diversity_samples.append(diversity)

            # Check for new best
            if current_best < best_energy:
                best_energy = current_best
                best_idx = np.argmin(e_np)
                best_clique = pop_gpu[best_idx].get().tolist()

                # Save clique if perfect
                if best_energy == 0:
                    clique_hash = clique_to_hash(best_clique)
                    if clique_hash not in found_cliques:
                        found_cliques.add(clique_hash)
                        clique_file = f"{run_dir}/clique_{clique_hash[:8]}.json"
                        with open(clique_file, 'w') as f:
                            json.dump({
                                'clique': best_clique,
                                'iteration': iteration,
                                'hash': clique_hash
                            }, f, indent=2)

                if convergence_iter is None and best_energy <= 5:
                    convergence_iter = iteration

                stag_global = 0
            else:
                stag_global += 500

            # Volcanic reheating
            if stag_global > REHEAT_THRESHOLD:
                temperature = REHEAT_TEMPERATURE
                stag_global = 0
            else:
                temperature = max(MIN_TEMPERATURE, temperature * COOLING_RATE)

            # Early stop if perfect clique found
            if best_energy == 0:
                break

    total_time = time.time() - start_time

    # Save results
    result = {
        'run_id': run_id,
        'success': best_energy == 0,
        'best_energy': best_energy,
        'convergence_iter': convergence_iter if best_energy == 0 else None,
        'total_iterations': iteration + 1,
        'total_time': total_time,
        'initial_best': initial_best,
        'unique_cliques': len(found_cliques),
        'history': history,
        'diversity': diversity_samples
    }

    with open(f"{run_dir}/results.json", 'w') as f:
        json.dump(result, f, indent=2)

    return result

# ============================================================================
# MAIN EXECUTION - 30 RUNS
# ============================================================================
print(f"\n{'='*70}")
print(f"🚀 RUNNING {N_RUNS} COLD START EXPERIMENTS")
print(f"   Max iterations per run: {MAX_ITERATIONS:,}")
print('='*70)

# Create results directory
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
results_dir = f"/content/drive/MyDrive/keller6_stats_{timestamp}"
os.makedirs(results_dir, exist_ok=True)

# Save configuration
config = {
    'n_runs': N_RUNS,
    'max_iterations': MAX_ITERATIONS,
    'population_size': POP_SIZE,
    'migration_threshold': MIGRATION_THRESHOLD,
    'exploration_prob': EXPLORATION_PROB,
    'reheat_threshold': REHEAT_THRESHOLD,
    'timestamp': timestamp
}
with open(f"{results_dir}/config.json", 'w') as f:
    json.dump(config, f, indent=2)

# Run experiments
all_results = []
start_time = time.time()

for run_id in range(N_RUNS):
    print(f"\n{'#'*60}")
    print(f"# RUN {run_id+1}/{N_RUNS}")
    print(f"{'#'*60}")

    # Set different seed for each run
    np.random.seed(42 + run_id)
    cp.random.seed(42 + run_id)

    result = run_single_experiment(run_id, results_dir)
    all_results.append(result)

    # Progress update
    elapsed = time.time() - start_time
    avg_time = elapsed / (run_id + 1)
    remaining = avg_time * (N_RUNS - run_id - 1)

    success_rate = sum(r['success'] for r in all_results) / (run_id + 1) * 100
    print(f"\n📊 Current stats after {run_id+1} runs:")
    print(f"   Success rate: {success_rate:.1f}%")
    print(f"   Avg time/run: {avg_time/60:.1f} min")
    print(f"   ETA: {remaining/60:.1f} min")

# ============================================================================
# FINAL STATISTICS
# ============================================================================
print(f"\n{'='*70}")
print("📊 FINAL STATISTICS")
print('='*70)

# Compile results
successes = [r for r in all_results if r['success']]
success_rate = len(successes) / N_RUNS * 100

convergence_iters = [r['convergence_iter'] for r in successes if r['convergence_iter']]
mean_convergence = np.mean(convergence_iters) if convergence_iters else None
std_convergence = np.std(convergence_iters) if convergence_iters else None

best_energies = [r['best_energy'] for r in all_results]
mean_best = np.mean(best_energies)
std_best = np.std(best_energies)

total_time = time.time() - start_time
unique_cliques = sum(r['unique_cliques'] for r in all_results)

print(f"\n🏆 SUCCESS RATE: {success_rate:.1f}% ({len(successes)}/{N_RUNS})")
if mean_convergence:
    print(f"⏱️  Mean convergence: {mean_convergence:.0f} ± {std_convergence:.0f} iterations")
print(f"📈 Mean best energy: {mean_best:.2f} ± {std_best:.2f}")
print(f"🎯 Unique cliques found: {unique_cliques}")
print(f"⏰ Total time: {total_time/3600:.2f} hours")

# Create summary table
summary = pd.DataFrame([
    {
        'Run': r['run_id'],
        'Success': '✓' if r['success'] else '✗',
        'Best Energy': r['best_energy'],
        'Convergence Iter': r['convergence_iter'] if r['convergence_iter'] else '-',
        'Total Iter': r['total_iterations'],
        'Time (min)': r['total_time']/60,
        'Unique Cliques': r['unique_cliques']
    }
    for r in all_results
])

print("\n📋 Detailed Results:")
print(summary.to_string(index=False))

# Save summary
summary.to_csv(f"{results_dir}/summary.csv", index=False)

# Plot results
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Success rate bar
axes[0,0].bar(['Success', 'Failure'], [success_rate, 100-success_rate],
              color=['green', 'red'], alpha=0.7)
axes[0,0].set_ylabel('Percentage (%)')
axes[0,0].set_title('Success Rate (0 conflicts)')
axes[0,0].grid(True, alpha=0.3)

# Best energy distribution
axes[0,1].hist(best_energies, bins=20, alpha=0.7, color='blue', edgecolor='black')
axes[0,1].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[0,1].set_xlabel('Best Energy Achieved')
axes[0,1].set_ylabel('Frequency')
axes[0,1].set_title('Distribution of Best Energies')
axes[0,1].grid(True, alpha=0.3)

# Convergence times
if convergence_iters:
    axes[1,0].hist(convergence_iters, bins=15, alpha=0.7, color='green', edgecolor='black')
    axes[1,0].set_xlabel('Convergence Iteration')
    axes[1,0].set_ylabel('Frequency')
    axes[1,0].set_title('Convergence Time Distribution')
    axes[1,0].grid(True, alpha=0.3)

# Progress over runs
cumulative_success = np.cumsum([1 if r['success'] else 0 for r in all_results])
axes[1,1].plot(range(1, N_RUNS+1), cumulative_success / range(1, N_RUNS+1) * 100,
              'b-o', linewidth=2, markersize=6)
axes[1,1].axhline(y=100, color='g', linestyle='--', alpha=0.5)
axes[1,1].set_xlabel('Number of Runs')
axes[1,1].set_ylabel('Cumulative Success Rate (%)')
axes[1,1].set_title('Learning Curve')
axes[1,1].grid(True, alpha=0.3)
axes[1,1].set_ylim(0, 105)

plt.tight_layout()
plt.savefig(f"{results_dir}/statistics.png", dpi=150)
plt.show()

print(f"\n💾 All results saved to: {results_dir}")
print(f"\n{'★'*60}")
print("🌟 STATISTICAL VALIDATION COMPLETE! 🌟")
print(f"{'★'*60}")

Mounted at /content/drive
📊 STATISTICAL VALIDATION - 30 COLD START RUNS

📊 Loading Keller-6 graph data...
✅ NAM loaded: 2.10 MB

⚡ Compiling global kernel...
✅ Kernel compiled

🚀 RUNNING 30 COLD START EXPERIMENTS
   Max iterations per run: 200,000

############################################################
# RUN 1/30
############################################################

📊 Current stats after 1 runs:
   Success rate: 100.0%
   Avg time/run: 6.7 min
   ETA: 195.2 min

############################################################
# RUN 2/30
############################################################

📊 Current stats after 2 runs:
   Success rate: 100.0%
   Avg time/run: 6.4 min
   ETA: 180.1 min

############################################################
# RUN 3/30
############################################################

📊 Current stats after 3 runs:
   Success rate: 100.0%
   Avg time/run: 6.0 min
   ETA: 162.0 min

#######################################################